## S2.1 — Spark Cluster Architecture
**Date completed:** May 2025  
**Status:** In Progress  
**Interview covered:** Q22, Q23 — Driver, Worker nodes, Master-Slave model

## Spark Cluster Architecture

### Three Components

**1 — Driver Node (Head Chef)**
- Receives your code from the notebook
- Creates execution plan (DAG)
- Splits work into tasks
- Sends tasks to executors
- Collects results and returns to notebook
- SparkSession lives here

**2 — Executor Nodes (Line Chefs)**
- Receive tasks from Driver
- Process data in parallel
- Each executor has CPU cores + RAM + local disk
- Send results back to Driver

**3 — Cluster Manager (Restaurant Owner)**
- Decides which machines to use
- Handles resource allocation
- In Databricks → Databricks manages this
- In standalone → YARN or Kubernetes

### Architecture Flow

### Interview Answers
**Q22 — Driver vs Worker node:**
Driver = brain, coordinates, plans, collects results.
Worker = muscle, executes tasks on actual data in parallel.

**Q23 — Master-Slave model:**
Driver = Master (1 per cluster), Executors = Slaves (many per cluster).
Master tells Slaves what to do. Slaves report back when done.

In [0]:
# S2.1 — Seeing Driver and Executors in action

# Step 1 — Check SparkSession (lives on Driver)
print("=== SparkSession Info ===")
print(f"Spark version: {spark.version}")
print(f"Compute: Serverless (Spark Connect)")

# Step 2 — Create data and see how Spark distributes it
print("\n=== Creating distributed data ===")
# Create a DataFrame with 1 million numbers distributed across partitions
df = spark.range(0, 1000000)

print(f"Total records: {df.count()}")

# Step 3 — Show data is distributed across executors
from pyspark.sql.functions import spark_partition_id, count

print("\n=== Partition Distribution ===")
partition_counts = df.groupBy(spark_partition_id().alias("partition_id")).agg(count("*").alias("record_count"))
partition_counts.show()

print("\nThis shows data is split across multiple executor partitions for parallel processing!")

In [0]:
# S2.1 — Key insight — parallel processing proof
print("=" * 50)
print("SPARK PARALLEL PROCESSING PROOF")
print("=" * 50)
print(f"Total records    : 1,000,000")
print(f"Total partitions : 8")
print(f"Records per partition: 125,000")
print(f"All 8 partitions processed SIMULTANEOUSLY")
print(f"This is why Spark handles big data efficiently")
print("=" * 50)

## Key Takeaways — S2.1

### 3 Components of Spark Cluster
| Component | Role | Count |
|-----------|------|-------|
| Driver node | Brain — plans, coordinates, collects | 1 per cluster |
| Executor nodes | Muscle — process data in parallel | Many per cluster |
| Cluster Manager | Infrastructure — resource allocation | 1 per cluster |

### Parallel Processing Proof
- 1,000,000 records split into 8 partitions
- 8 executors processed simultaneously
- Each partition = 125,000 records
- All 8 ran at the SAME TIME — that is Spark's power

### Serverless vs Classic
- Serverless = DataFrame API only (Spark Connect)
- Classic = supports RDD + DataFrame
- 2025 standard = DataFrame API always preferred

### Interview Answers
**Q22:** Driver = Master (1) coordinates. Executor = Worker (many) processes data.  
**Q23:** Master-Slave model — Driver gives tasks, Executors execute them.

### Next
S2.2 — Spark Execution Model — how Driver breaks your code into Jobs → Stages → Tasks

## Live Proof — Query Profile Analysis

### Execution Flow (bottom to top)
| Step | Operation | Rows | What happened |
|------|-----------|------|---------------|
| #6 | Range | 1M | Generated 1M rows in 8 partitions |
| #5 | Aggregate | 8 | Each executor counted its partition locally |
| #4 | Shuffle | 8 | 8 partial counts MOVED across network |
| #3 | Aggregate | 1 | 8 counts combined into final total |
| #2 | Columnar to Row | 1 | Format conversion for display |
| #1 | Result | 1 | Returned to your screen |

### Key Findings
- Shuffle = most expensive step (34ms, 32MB memory)
- Shuffle = data movement BETWEEN executors — not calculation
- Calculation happens in Aggregate steps
- Bytes read = 0 because spark.range() is in-memory generation

### Shuffle vs Partitioning
- Partitioning = data split across executors (no network movement)
- Shuffle = data MOVES between executors (network travel = expensive)

# How to reduce shuffle cost:

Strategy 1 — Broadcast Join (S4.3)
Small table → send full copy to every executor
No shuffle needed — each executor already has the data

Strategy 2 — Reduce Shuffle Partitions (S2.12)
Default shuffle partitions = 200
For small data → too many = overhead
spark.conf.set("spark.sql.shuffle.partitions", "8")
Match to your actual data size

Strategy 3 — Filter Early (S2.5)
Filter data BEFORE groupBy
Fewer rows to shuffle = less network travel

Strategy 4 — Partition Data Wisely (S2.11)
Pre-partition by the column you will groupBy
Data already in right place = less shuffling needed

Strategy 5 — Avoid Unnecessary groupBy
Every groupBy = shuffle
Only group when absolutely needed
